In [38]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load dataset

In [ ]:
def load_data(file_path):
    try:
        data = pd.read_csv(file_path)
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None
    
df = load_data("../../Database/PDF/Customer-Churn-dataset.csv")

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339


## Fraud logic (Rule-Based)

In [40]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

# Create fraud logic on condition
import numpy as np
conditions = (
    (df["Balance"] > 150000) &
    (df["NumOfProducts"] <=1 ) &
    (df["IsActiveMember"] == 0)
) | (
    (df["CreditScore"] < 400) &
    (df["Balance"] > 100000)
) | (
    (df["Complain"] > 0) &
    (df["Satisfaction Score"] <= 2)
)

df["Fraud"] = np.where(conditions, 1, 0)

In [41]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377,0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564,0
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339,1


In [42]:
# reducte fraud cases aritifically
fraud_idx = df[df["Fraud"] == 1].sample(frac=0.3, random_state=42).index
df["Fraud"] = 0
df.loc[fraud_idx, "Fraud"] = 1

## Add advanced fraud features

In [43]:
# High risk score (engineered features)
df["RiskScore"] = (
    (df["CreditScore"] < 500).astype(int) +
    (df["Balance"] > 100000).astype(int) +
    (df["IsActiveMember"] == 0).astype(int) +
    (df["Complain"] > 0).astype(int)
)

# Balance per product
df["BalancePerProduct"] = df["Balance"] / (df["NumOfProducts"] + 1)

# Age risk bucket
df["AgeRisk"] = np.where(df["Age"] < 25, 1, np.where(df["Age"] > 60, 1, 0))

In [44]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464,0,1,0.000000,0
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456,0,1,41903.930000,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377,0,3,39915.200000,0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350,0,1,0.000000,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425,0,1,62755.410000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300,0,1,0.000000,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771,0,0,28684.805000,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564,0,1,0.000000,0
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339,1,2,25025.103333,0


In [45]:
print(df["Fraud"].value_counts(normalize=True))

0    0.9668
1    0.0332
Name: Fraud, dtype: float64


## Add features for Market Risk

In [46]:
df["HighValueCustomer"] = (df["Balance"] > 100000).astype(int)
df["LowCreditRisk"] = (df["CreditScore"] < 500).astype(int)

# Mkarketing score
"""Marketing score: combines high value customer status, low credit risk, and non-France geography to identify customers 
who may be more receptive to marketing campaigns, which can help target efforts effectively."""
df["MarketingScore"] = (
    df["HighValueCustomer"] + df["LowCreditRisk"] + (df["Geography"] != "France").astype(int)
).drop(columns=["HighValueCustomer", "LowCreditRisk"])

## Operation Risk feature for internal banking

In [47]:
df["ComplainFlag"] = (df["Complain"] > 0).astype(int)
df["LowSatisfaction"] = (df["Satisfaction Score"] <= 2).astype(int)

# Systemic risk score
"""Operational risk score: combines complaint and satisfaction flags with inactivity to identify customers at risk of leaving or being dissatisfied, 
which can impact operational stability."""
df["OperationalRiskScore"] = (
    df["ComplainFlag"] +
    df["LowSatisfaction"] +
    (df["IsActiveMember"] == 0).astype(int)
).drop(columns=["ComplainFlag", "LowSatisfaction"])

In [48]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,...,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,France,Female,42,2,0.00,1,1,1,101348.88,...,0,1,0.000000,0,0,0,0,1,1,2
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,...,0,1,41903.930000,0,0,0,1,1,0,1
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,...,0,3,39915.200000,0,1,0,1,1,0,2
3,699,France,Female,39,1,0.00,2,0,0,93826.63,...,0,1,0.000000,0,0,0,0,0,0,1
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,...,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,...,0,1,0.000000,0,0,0,0,0,1,2
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,...,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,...,0,1,0.000000,0,0,0,0,1,0,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,...,1,2,25025.103333,0,0,0,1,1,1,3
